#### Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import pyprind
import scipy
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import pickle
import sys

import brightway2 as bw
import presamples as ps 
import pelicun_2_bw2 as p2b

### Select the Brightway2 project

In [ ]:
project_name = 'Brightcon_demo_stochastic_inputs'

if project_name not in [x.name for x in bw.projects]:
    bw.restore_project_directory(fp="Brightcon_demo_stochastic_inputs.tar.gz")
    bw.projects.set_current(project_name)
else : 
    bw.projects.set_current(project_name)
print(bw.projects.current)

In [ ]:
list(bw.databases)

#### Load damage samples

In [ ]:
# Load the samples:
fp_dmgs = os.path.join(os.getcwd(),'case_study_inputs','dmg_samples_MWE.pickle')
with open(fp_dmgs, 'rb') as f:
    detailed_dmgs = pickle.load(f) 
# Indicate the content of the samples:
for MRI in detailed_dmgs.keys() : 
    print(MRI,detailed_dmgs[MRI].shape[0])  
ls_MRIs = list(detailed_dmgs.keys())

# Complementary information:
loss_map_fp = os.path.join(os.getcwd(),'case_study_inputs','loss_map.xlsx')
loss_map = pd.read_excel(loss_map_fp,index_col=0)

In [ ]:
display(detailed_dmgs[ls_MRIs[0]])

#### Retrieve the presampled LCA scores

In [ ]:
# Folder where the preaggregated generic losse results are sent : 
target_directory = os.path.join(os.getcwd(),'dbmc_output_dir')
generic_loss_db = 'db_name'
preagg_db = 'Building_presampled_repairs'

# Path leading to DBMC final results :
lcia_fp = os.path.join(target_directory,generic_loss_db,"results","LCIA")

In [ ]:
# Selected LCIA methods
list_of_methods = [('IPCC 2021', 'climate change: total (excl. biogenic CO2)', 'global warming potential (GWP100)'),('IMPACT World+ Damage 2.2.1 for ecoinvent v3.12','Ecosystem quality','Total ecosystem quality'),('IMPACT World+ Damage 2.2.1 for ecoinvent v3.12','Human health','Total human health')]


### Format the first presample

<div align="center">
<img src="imgs/background_presample.png" style="width: 50%;">
</div>



In [ ]:
# Variable to load DBMC results
arrays_list = []
# Simulatenously list the datasets to which individual simulations belong : 
indices = []

# Start extracting DBMC results, matching preaggregrated LCIA results (as fake elementary flows) with LCIs 
for m in list_of_methods : 
    for act in bw.Database(preagg_db) : 
        arrays_list.append(np.load(os.path.join(lcia_fp, bw.Method(m).get_abbreviation(), act.key[1]+'.npy')).reshape(-1,1))
        indices.append(
            (
            ('biosphere3', bw.Method(m).get_abbreviation()),
                act.key,
                'biosphere'
            ))

# Stack the arrays
samples = np.hstack(arrays_list)

# Transpose
samples = samples.T
print(f'The shape of the loaded datapackage is : {samples.shape}')

# Final formatting :  
label = 'biosphere'
agg_matrix_data = [(samples, indices, label)]

# Loading to presamples : 
ps_id, ps_fp = ps.create_presamples_package(matrix_data=agg_matrix_data)
ps_fp

#### Format the second presampling layer : the sequential sampling tables

<div align="center">
<img src="imgs/Foreground_presample.png" style="width: 50%;">
</div>


In [ ]:
foreground_loss_db_name = 'DS_LCI-Building'

# Create individual foreground databases assessing the wind damage scenarios :
pp_paths = []

for run in ls_MRIs : 
    foreground_name = f'{foreground_loss_db_name}-Wind-MRI-{run}'
       
    # Create linking between pre-aggregated damage states and foreground inventories : 
    p2b.LCA_setup.linking_DS_to_LCI(foreground_name,preagg_db,detailed_dmgs[run],loss_map)
    # Whilst there, also pre-set presamples :
    pp_paths.append(p2b.LCA_setup.presamples_setup(foreground_name,preagg_db,detailed_dmgs[run],loss_map)) 


## Run the dual-presampling

<div align="center">
<img src="imgs/Sample_evaluation.png" style="width: 50%;">
</div>


### Aggregated results

In [ ]:
%%time

sample_size = 12000 # 12 000 samples, per wind speed
print(f'Generating {sample_size} samples with {len(list_of_methods)} indicators for {len(ls_MRIs)} wind speeds.')

# Run the LCA

# Calculation setups :
iterations = sample_size
LCIA_methods = list_of_methods

# Storage variable :
losses = {}

# Start iterating the MRIs losses : 
for j,run in enumerate(ls_MRIs) :
    
    # Preparing the reference flow and matrices :
    foreground_name = f'{foreground_loss_db_name}-Wind-MRI-{run}' 
    demand = {(foreground_name, 'Structure losses'): 1} # To populate the demand vector
    if j == 0 : # Ensure this is only ran on the first iteration
        C_matrices = p2b.Presample_LCA_calculations.get_C_matrices(demand,LCIA_methods) # Pre-load the characterization matrices
    mc_scores = np.empty(shape=[len(LCIA_methods), iterations]) # Pre-organize a matrix to store results.

    # Instantiate an object for MonteCarlos : 
    mc_ps = bw.MonteCarloLCA(
        demand,
        presamples=[

                    ps_fp,
                    pp_paths[j],
                   ]
    )

    # Launch calculations :
    print(f'Assessing wind MRI {run}, with {iterations} simulations.')

    for iteration in pyprind.prog_bar(range(iterations), stream=sys.__stdout__):
        lci = next(mc_ps)  
        for i,m in enumerate(LCIA_methods):
            mc_scores[i, iteration] = (C_matrices[m]*mc_ps.inventory).sum()

    # Force reset the sequential indexer
    mc_ps.presamples.reset_sequential_indices()
    # Clear the variable before giving it another run
    del mc_ps 
    losses[run] = mc_scores
    print('')

### Visuals

In [ ]:
# Clear some faulty MC simulations that yield unrealistic results : 

# Storage variable
cleansed_losses = {}

for results in losses : 
    lwr_bnd = -1*10**(-5)  # Sometimes a simulation might be a true 0 (or a true close to 0)
    upper_percentile_bnd = 99.5 # With large simulation counts, some simulations return unrealisticly high results
                              # results may thus be cleansed by removing upper percentiles (from 0th to 100th)
    # Remove samples below the lower bound :
    arr = losses[results]
    cond = np.any(arr<lwr_bnd,axis=0)
    filtered_arr = arr.T[~cond]

    # Remove samples above the percentile treshold (through a master array)
    ms_ls = []
    for val in range(len(LCIA_methods)):    
        uppr_bnd = np.percentile(filtered_arr[:,val],upper_percentile_bnd)
        
        # Make all value lower than the threshold as "TRUE"
        cond = filtered_arr[:,val]<uppr_bnd
        ms_ls.append(cond)
    
    # Make a unique array flagging anytime a FALSE is detected within the samples : 
    ms_cond = np.logical_and.reduce((ms_ls))
    re_filtered_arr = filtered_arr[ms_cond]    
    cleansed_losses[results] = re_filtered_arr

# Visual outputs
fig, axs = plt.subplots(len(ls_MRIs), len(LCIA_methods), figsize=(20, 6*len(ls_MRIs)))

for run_id,results in enumerate(cleansed_losses.keys()) :
    for val in range(len(LCIA_methods)) :
        
        arr = cleansed_losses[results][:,val]
        
        # Format the results
        _, bin_edges = np.histogram(arr,density = True)


        # Plot the results
        axs[run_id,val].hist(arr,bins = bin_edges,alpha = 0.3)  

        # Add figure descriptions :
        if run_id == 0 : 
            axs[run_id,val].set_title(f'{LCIA_methods[val][0]}-{LCIA_methods[val][2]}')
        if val == 0 :
            axs[run_id,val].set_ylabel(f'Count at MRI {results}')###############
        if run_id == len(cleansed_losses.keys())-1 : 
            axs[run_id,val].set_xlabel(bw.Method(LCIA_methods[val]).metadata['unit'])

In [ ]:
df = pd.DataFrame(losses[ls_MRIs[0]]).T
df.columns= list_of_methods
display(df)

In [ ]:
display(detailed_dmgs[ls_MRIs[0]])